In [3]:
%pip install --upgrade numpy

Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.7/15.7 MB 181.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow 2.17.0 requires numpy<2.0.0,>=1.26.0; python_version >= "3.12", but you have numpy 2.5.5 which is incompatible.
scipy 1.14.1 requires numpy<2.3,>=1.23.5, but you have numpy 2.5.5 which is incompatible.

[notice] A new release of pip is available: 24.2 -> 26.3
[notice] To update, run: python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [4]:
%pip install torch pyarrow

Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 539.2/539.2 MB 105.4 MB/s eta 0:00:00m eta 0:00:010:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 519.2/519.2 MB 104.6 MB/s eta 0:00:00m eta 0:00:010:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.1/170.1 MB 136.2 MB/s eta 0:00:00m eta 0:00:010:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.0/216.0 MB 144.1 MB/s eta 0:00:00m eta 0:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.4/135.4 MB 142.9 MB/s eta 0:00:00m eta 0:00:010:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 404.9/404.9 MB 119.3 MB/s eta 0:00:00m eta 0:00:010:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 149.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.0/47.0 MB 152.3 MB/s eta 0:00:00m eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 149.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 218.3/218.3 MB 127.9 MB/s

In [7]:
#!/usr/bin/env python3
# pylint: disable=line-too-long

"""
Feed-forward network to predict a player's pre-flop action (fold, check/call, raise).
Just an evaluation, not feature dense.

High Level Approach
1. Load 10M pre-flop decisions from parquet file
2. Turn each row into numbers the network can use (features + hand id)
3. Split the data into Train(90%) vs Test(10%)
4. build a neural network : inputs -> hidden -> output (3)
5. Train : guess, measure error, adjust weights, repeat
5. Test it on rows it has never trained before

Reads pre-flop rows from the decisions/ table written by phh_to_parquet.py. Each row becomes 12 numbers:
    cards | known flag, high rank, low rank, suited, pair (all 0 when cards are hidden)
    table | seat (0 = small blind), n_players, n_active, n_raises
    chips | pot, to_call, and stack

Basic usage:
    ./scripts/feed_forward_preflop.py
"""

##### other TODOs ####################
# TODO: not attached to any one part of the code, just improvements to make:
# Infer hidden cards based on current representation distributions and what action the player took for better training quality.
#     most significantly, need to unroll the unknown's fold rates back into the weaker hands
#     (Kristian)
#
# Use an alternative hand encoding that better represents how strong the hands are sensitive to how people play them.
#     lets the model learn card strengths based on how people play rather than the mathematical significance
#     (Surja)
#
# Use all of the data, stream the rows instead of loading it all into memory.
#     should make this optional, so we can test with a smaller sample to get the other features implemented first
#     (Cole)
#
# Include player IDs into the decisions/ table and model learning
#     think this would be useful as a PoC that it can adapt to new players/data with non-traditional behavior
#     (Kirsten)
#
# More hidden layers & epochs
#     (Surja)
######################################

# std libs
import sys
import random
from pathlib import Path
from argparse import ArgumentParser

# external libs
import torch
import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.parquet as pq

##### constants ####################
ACTION_NAMES = ["fold", "call", "raise"] # class 0,1,2 models 3 outputs
#columns pulled out of parquet files, anything not listed is skipped
COLUMNS = ["hand_id", "seat", "n_players", "n_active", "big_blind", "pot", "to_call", "stack", "n_raises", "hole1", "hole2", "action"]

# pre-flop only, and skip the bad rows: some sites record unknown stacks as inf, and negative blind entries make the pot negative
# TODO: fix or remove these bad rows...
#     (Cole)
# Keep a row only when this conditions are true
# pre-flop only(no flop/turn/river decisions)
# drop rows when a site recorded unknown stack as inf
# drop rows with negative blindsm which give a negative pot
ROW_FILTERS = (pc.field("street") == 0) & pc.is_finite(pc.field("stack")) & (pc.field("pot") >= 0)

##### funcs ####################
def get_args():
    """ Argparser retrieval """

    parser = ArgumentParser(description=__doc__)

    parser.add_argument("--data_path", type = Path, help = "path to the decisions parquet table", default = "parquet/decisions")
    parser.add_argument("--max_rows", type = int, help = "pre-flop rows to load", default = 10000000) # TODO: change to optional for final model (Cole)
    parser.add_argument("--epochs", type = int, help = "dataset iterations", default = 3) # one full pass
    parser.add_argument("--threads", type = int, help = "torch threads", default = 2) # 2 seems to be safe on the VMs
    # set a different data path to read parquet, below values overide default conditions above with 10 epochs and 8 threads
    return parser.parse_args(["--data_path", "../parquet/decisions", "--epochs", "10", "--threads", "8"])


def load_preflop(data_path, max_rows):
    """ Reads pre-flop rows from randomly ordered parts until we have max_rows return """
    # TODO: fairly sure there's a better way to do this leveraging pyarrow, to research...
    #     (Kirsten)

    tables = [] #one pyarrow tables per file
    n_rows = 0 #running total rows added 
 
    files = sorted(data_path.rglob("*.parquet")) #finding every parquet file in the data path

    # shuffling the original dataset organization so they aren't grouped by stake
    # should remove the fixed seed if we don't want to change other configurations like Adam/ReLU
    random.Random(0).shuffle(files)

    for path in files:
        #only read the columns we need and only the rows which passes ROW_FILTERS condition 
        tables.append(pq.read_table(path, columns = COLUMNS, filters = ROW_FILTERS))
        n_rows += len(tables[-1]) # num rows in current table
        if n_rows >= max_rows: #stop reading files when mx_rows condition reached
            break
    table = pa.concat_tables(tables).slice(0, max_rows) # trim trailing rows beyond the max_rows

    return { # parquet -> numpy -> Tensor based dict all in one, didn't see a direct parquet -> Tensor util
        column: torch.from_numpy(table[column].to_numpy()) for column in COLUMNS
    }

def make_hand_index(columns): # Hand Index feeds the embedding
    """ Turn two holes cards into ONE Integer ID, so the embedding can look it up """

    # Cards are numbered 0..51 as: rank * 4 + suit
    # rank 0..12 = 2,3,.....K,A
    # suit 0..3 = c,d,h,s Example: As = 12*4 + 3 = 51
    hole1 = columns["hole1"].long() # long() = whole number tensor
    hole2 = columns["hole2"].long() # indexing into an embedding
    rank1,rank2 = hole1//4, hole2//4 # 4 drops the suit and leaves the rank
    
    #sort so order of two card doesnt matter (A-K and K-A are same hand)
    high=torch.maximum(rank1,rank2)
    low = torch.minimum(rank1,rank2)

    # %4 leaves the suit. same suit -> 1(suited), else 0(offsuit)
    suited = ((hole1%4) == (hole2%4)).long()

    #combine high, low, suited into one unique index number
    index = high * 26 + low * 2 + suited

    #hidden cards are stored as -1  where card are known have a real number otherwise use special id 338
    return torch.where(hole1 >= 0, index, torch.full_like(index,338))
    

def make_features(columns):
    """ Raw decision columns -> (rows, 12) float tensor """

    # in-house ref for card encodings:
    #   rank * 4 + suit
    #   - ranks 2 -> A as 0 -> 12 and suits c d h s as 0 -> 3.
    #       2c = 0  * 4 + 0 = 0
    #       2s = 0  * 4 + 3 = 3
    #       8d = 6  * 4 + 1 = 25
    #       As = 12 * 4 + 3 = 51

    hole1 = columns["hole1"]
    hole2 = columns["hole2"]
    known = hole1 >= 0

    # flooring gives us ranks; e.g. aces // 4 = 12
    rank1 = hole1 // 4
    rank2 = hole2 // 4

    big_blind = columns["big_blind"]

    # NOTE: organizing with dim = 1, rotates the decisions rows to append horizontally in the vector space
    return torch.stack([
        # card features; * known seeds 0s when the cards are hidden
        torch.maximum(rank1, rank2) / 12 * known, # always seed high card first
        torch.minimum(rank1, rank2) / 12 * known,
        ((hole1 % 4) == (hole2 % 4)) * known, # suited flag
        (rank1 == rank2) * known, # pair flag

        known, # flag
        # all divided by a scaling factor to get them between 0-1
        # should probably resolve this instead of going off the current dataset maximums
        columns["seat"] / 9,
        columns["n_players"] / 10,
        columns["n_active"] / 10,
        columns["n_raises"],

        # dividing to get the chip values as a relation to the big blind; idea from Cole
        # log1p is essentially "safe log" for the 0 case; using log to prevent the inevitable squish the same divising factor logic from before would provide
        #   since the vast majority of values in this dataset are less than the big blind, outliers would result in strong biases'
        # TODO: should test other normalization strategies to see if those make a significant difference
        #     (Kristian - deprioritized)
        torch.log1p(columns["pot"] / big_blind),
        torch.log1p(columns["to_call"] / big_blind),
        torch.log1p(columns["stack"] / big_blind),
    ], dim = 1)


def print_example_hands(model):
    """ Sanity check/hand samples: first to act in a 6 player game, nobody has raised, 100 big blind stacks """

    # put high card to the left, not resolving high card like in make_features
    sample_hands = {
        "AA": (51, 50),
        "AKs": (51, 47),
        "JTs": (39, 35),
        "72o": (20, 1),
        "??": (-1, -1)
    }

    n = len(sample_hands)

    # NOTE: .full((n,) ...) looks awkward but full expects a tuple or list and we only need 1D
    # https://docs.pytorch.org/docs/2.15/generated/torch.full.html
    columns = {
        "hole1": torch.tensor([hand[0] for hand in sample_hands.values()]),
        "hole2": torch.tensor([hand[1] for hand in sample_hands.values()]),
        "seat": torch.full((n,), 2), # first player after sm/bb
        "n_players": torch.full((n,), 6), # 6 players
        "n_active": torch.full((n,), 6),
        "n_raises": torch.zeros(n), # no raises
        "big_blind": torch.ones(n),
        "pot": torch.full((n,), 1.5), # just small + big blind
        "to_call": torch.ones(n),
        "stack": torch.full((n,), 100.0) # buy in default
    }

    with torch.no_grad():
        probablities = model(make_features(columns), make_hand_index(columns)).softmax(dim = 1)

    print("\nSelect hands sample")
    print(f"\t{"hand":<5}" + "".join(f"{action:>8}" for action in ACTION_NAMES))
    for hand, hand_probabilities in zip(sample_hands, probablities):
        print(f"\t{hand:<5}" + "".join(f"{prob:>8.2f}" for prob in hand_probabilities.tolist()))


class PreflopNet(torch.nn.Module):
    def __init__(self,n_features,n_hands = 339,embed_dim = 8):
        super().__init__()
        self.hand_embedding = torch.nn.Embedding(n_hands,embed_dim)
        self.mlp = torch.nn.Sequential(
            torch.nn.Linear(n_features+embed_dim, 128),
            torch.nn.ReLU(), # pretty much just using the most common features (which the baove source uses), ReLU happens to be suitable for this application
            torch.nn.Linear(128, 128),
            torch.nn.ReLU(),
            torch.nn.Linear(128, 128),
            torch.nn.ReLU(),
            torch.nn.Linear(128, 64),
            torch.nn.ReLU(),
            torch.nn.Linear(64, len(ACTION_NAMES)),
        )

    def forward(self,features,hands):
        return self.mlp(torch.cat([features,self.hand_embedding(hands)],dim = 1))


def main():
    """ main driver - loads, trains, & reports """
    # TODO: convert most of this to a proper class structure, this isn't great
    #     (Kristian)

    args = get_args()
    torch.manual_seed(0) # fixed for now, could probably be an arg
    torch.set_num_threads(args.threads)

    columns = load_preflop(args.data_path, args.max_rows)
    features = make_features(columns)
    hands = make_hand_index(columns) #New addition

    labels = columns["action"].long() # cross_entropy() requires long/byte, fails without
    known = columns["hole1"] >= 0 # -1 indicates unknown

    # splitting test/training subsets, just pulling about 1 in 10 for testing
    test_subset = columns["hand_id"] % 10 == 0
    train_in, train_hands, train_target = features[~test_subset], hands[~test_subset], labels[~test_subset]
    test_in, test_hands, test_target = features[test_subset], hands[test_subset], labels[test_subset]

    print(f"Loaded {len(labels):,} pre-flop rows ({len(train_target):,} train, {len(test_target):,} test)")

    model = PreflopNet(features.shape[1])
    """# Sequential for linear cxts - https://docs.pytorch.org/docs/2.15/generated/torch.nn.Sequential.html
    model = torch.nn.Sequential(
        torch.nn.Linear(features.shape[1], 128),
        torch.nn.ReLU(), # pretty much just using the most common features (which the baove source uses), ReLU happens to be suitable for this application
        torch.nn.Linear(128, 128),
        torch.nn.ReLU(),
        torch.nn.Linear(128, 128),
        torch.nn.ReLU(),
        torch.nn.Linear(128, 64),
        torch.nn.ReLU(),
        torch.nn.Linear(64, len(ACTION_NAMES)),
    )"""



    

    # Tuning handler, Adam appears to just be the most common optimizer, can tune if noted to be a limiting factor
    # default lr is already 1e-3 (from lib: `float | Tensor = 1e-3`), specified it because we may want to tune this
    optimizer = torch.optim.Adam(model.parameters(), lr = 1e-3)

    for _ in range(args.epochs):
        # further shuffling to mostly isolate the individual games into different batches
        # TODO: re-evaluate how many steps to split into once we ingest the full dataset, probably needs to be a function of N
        #     (Cole)
        for index in torch.randperm(len(train_target)).split(4000):
            loss = torch.nn.functional.cross_entropy(model(train_in[index], train_hands[index]), train_target[index])
            optimizer.zero_grad()
            loss.backward() # backprop
            optimizer.step()

    # run a test to git the hit rate for accuracy
    with torch.no_grad(): # can more or less use no_grad() for all testing since we don't need to do backprop
        correct = model(test_in, test_hands).argmax(dim = 1) == test_target

    print(f"\n{"test results":<14}{"rows":>12}{"accuracy":>10}")
    print(f"{"combined":<14}{len(correct):>12,}{correct.float().mean().item():>10.3f}")
    for name, group in [("cards known", known), ("cards unknown", ~known)]:
        test_group = group[test_subset]
        print(f"{name:<14}{test_group.sum().item():>12,}{correct[test_group].float().mean().item():>10.3f}")

    print_example_hands(model)


if __name__ == "__main__":
    sys.exit(main())


Loaded 10,000,000 pre-flop rows (8,999,579 train, 1,000,421 test)


/home/coder/.local/lib/python3.12/site-packages/torch/autograd/graph.py:1113: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12060). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /__w/pytorch/pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass



test results          rows  accuracy
combined         1,000,421     0.773
cards known         59,941     0.803
cards unknown      940,480     0.771

Select hands sample
	hand     fold    call   raise
	AA       0.00    0.25    0.75
	AKs      0.00    0.12    0.88
	JTs      0.00    0.47    0.53
	72o      0.00    0.78    0.22
	??       0.84    0.02    0.14


SystemExit: 

/usr/local/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3585: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
